In [1]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 13.9 MB/s eta 0:00:00


In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset , DataLoader
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

In [3]:
torch.manual_seed(42)

In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [6]:
df = pd.read_csv('fashion-mnist_train.csv')

In [7]:
df

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,6,0,0,0,0,0,0,0,5,0,...,0,0,0,30,43,0,0,0,0,0
3,0,0,0,0,1,2,0,0,0,0,...,3,0,0,0,0,1,0,0,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59995,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
59996,1,0,0,0,0,0,0,0,0,0,...,73,0,0,0,0,0,0,0,0,0
59997,8,0,0,0,0,0,0,0,0,0,...,160,162,163,135,94,0,0,0,0,0
59998,8,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [8]:
X = df.iloc[:,1:].values
y = df.iloc[: , 0].values

In [9]:
x_train,x_test,y_train,y_test = train_test_split(X,y,test_size = 0.2,random_state=42)

In [10]:
x_train = x_train/255.0
x_test = x_test/255.0

In [11]:
class CustomDataset(Dataset):

  def __init__(self,features,labels):
    self.features = torch.tensor(features,dtype=torch.float32)
    self.labels = torch.tensor(labels,dtype=torch.long)

  def __len__(self):
    return len(self.features)

  def __getitem__(self,index):
    return self.features[index],self.labels[index]


In [12]:
train_dataset = CustomDataset(x_train,y_train)
test_dataset = CustomDataset(x_test,y_test)

In [13]:
train_loader = DataLoader(train_dataset , batch_size = 32,shuffle = True,pin_memory = True)
test_loader = DataLoader(test_dataset,batch_size = 32,shuffle=False,pin_memory=True)

In [14]:
len(train_loader)

1500

In [ ]:
# =============================================================================
# OPTUNA / BAYESIAN OPTIMIZATION — COMPLETE UNDERSTANDING
# =============================================================================
#
# 🧠 1) Why do we need Optuna?
#
# When training a model, we choose hyperparameters:
# - learning rate
# - dropout
# - L2 lambda
# - hidden neurons
# - batch size
# - optimizer type
#
# Example:
# lr = ?
# dropout = ?
# lambda = ?
# hidden_units = ?
#
# Different values give different validation accuracy.
#
# Example:
# lr = 0.1   → 70%
# lr = 0.01  → 85%
# lr = 0.002 → 93%
#
# Goal:
# Find hyperparameters that give maximum validation accuracy.
#
#
# -----------------------------------------------------------------------------
# 🧠 2) Hidden graph exists
# -----------------------------------------------------------------------------
#
# There is a hidden relationship:
#
# Hyperparameters → Validation Accuracy
#
# Mathematically:
#
# f(hyperparameters) = validation accuracy
#
# Example:
# f(lr=0.01, dropout=0.2)  = 86%
# f(lr=0.002, dropout=0.3) = 93%
#
# This function is unknown.
#
# We need to discover it.
#
#
# -----------------------------------------------------------------------------
# 🧠 3) Think of it like a graph / mountain
# -----------------------------------------------------------------------------
#
# For one hyperparameter:
#
# x-axis → learning rate
# y-axis → validation accuracy
#
# Graph:
#
# Accuracy
#   ^
# 93|               peak
# 91|            *
# 85|       *
# 70| *
#   +------------------------>
#
# Peak = best learning rate.
#
#
# For many hyperparameters:
# - learning rate
# - dropout
# - lambda
# - hidden units
#
# Graph becomes a multi-dimensional surface.
#
# Think:
# A mountain landscape.
#
# Goal:
# Find highest mountain peak.
#
# Highest peak = best hyperparameters.
#
#
# -----------------------------------------------------------------------------
# 🧠 4) Optuna starts with trials
# -----------------------------------------------------------------------------
#
# It tries some combinations:
#
# Trial 1 → accuracy = 70
# Trial 2 → accuracy = 85
# Trial 3 → accuracy = 91
#
# Now it has observations/data points.
#
#
# -----------------------------------------------------------------------------
# 🧠 5) It uncovers graph nature
# -----------------------------------------------------------------------------
#
# This is the key idea.
#
# Optuna studies previous trials and understands:
#
# - Which region is good?
# - Which region is bad?
# - Where might the peak be?
#
# Example:
#
# lr = 0.001 → 91
# lr = 0.002 → 93
# lr = 0.003 → 92
#
# Then Optuna thinks:
#
# Peak likely near 0.002
#
# This is called:
# Understanding graph nature.
#
#
# -----------------------------------------------------------------------------
# 🧠 6) It predicts for unexplored points
# -----------------------------------------------------------------------------
#
# For untested point:
#
# Example:
# lr = 0.0025
#
# Optuna predicts:
#
# Expected accuracy ≈ 92.5
#
# Meaning:
#
# Probably a good point.
#
#
# -----------------------------------------------------------------------------
# 🧠 7) Uncertainty (MOST IMPORTANT)
# -----------------------------------------------------------------------------
#
# For each untested point, Optuna asks:
#
# "How sure am I about this prediction?"
#
# That is uncertainty.
#
#
# Low uncertainty:
#
# Many nearby points tested.
#
# Example:
# 0.002 region is well explored.
#
# Prediction:
# Expected = 92.5 ± 1
#
# Meaning:
# Very confident.
#
#
# High uncertainty:
#
# Region never explored.
#
# Example:
# 0.05 region never tested.
#
# Prediction:
# Expected = 80 ± 20
#
# Meaning:
# Could be 60
# Could be 100
# Don't know
#
# This is uncertainty.
#
#
# -----------------------------------------------------------------------------
# 🧠 8) How Optuna chooses next point
# -----------------------------------------------------------------------------
#
# It considers TWO things:
#
# A) Promising region:
#
# Looks good.
#
# Example:
# Expected = 93
#
# Try nearby.
#
#
# B) Uncertain region:
#
# Unknown area.
#
# Example:
# Expected = 82
# But uncertainty is huge
# Maybe hidden peak exists
#
# Worth checking.
#
#
# So next point is chosen based on:
#
# Looks good
# +
# Worth exploring
#
#
# -----------------------------------------------------------------------------
# 🧠 9) Exploitation
# -----------------------------------------------------------------------------
#
# Means:
#
# Search near good point.
#
# Example:
#
# Best point:
# 0.002 → 93
#
# Next tries:
# 0.0025
# 0.0018
#
# Nearby search.
#
#
# -----------------------------------------------------------------------------
# 🧠 10) Exploration
# -----------------------------------------------------------------------------
#
# Means:
#
# Check unknown region.
#
# Example:
#
# 0.05 never tested
#
# Maybe hidden better peak exists.
#
# Check once.
#
#
# -----------------------------------------------------------------------------
# 🧠 11) Why uncertainty matters
# -----------------------------------------------------------------------------
#
# Without uncertainty:
#
# Optuna becomes greedy:
#
# Always search near current best.
#
# Problem:
# May miss global optimum.
#
#
# With uncertainty:
#
# Optuna thinks:
#
# Unknown region may hide better peak.
#
# So it explores.
#
# This avoids getting stuck in local optimum.
#
#
# -----------------------------------------------------------------------------
# 🧠 12) Final workflow
# -----------------------------------------------------------------------------
#
# Try few points
# ↓
# Observe accuracy
# ↓
# Understand graph shape
# ↓
# Estimate good / bad regions
# ↓
# Estimate uncertainty
# ↓
# Choose next point smartly
# ↓
# Move toward peak
# ↓
# Find best hyperparameters
#
#
# -----------------------------------------------------------------------------
# 🎯 FINAL MASTER UNDERSTANDING
# -----------------------------------------------------------------------------
#
# Optuna uses Bayesian Optimization to intelligently search hyperparameters.
#
# It:
# - performs trials
# - learns hidden relationship between hyperparameters and validation accuracy
# - understands promising regions
# - understands poor regions
# - estimates uncertainty in unexplored regions
# - chooses next point by balancing:
#       exploitation (search near good regions)
#       exploration (search uncertain regions)
#
# Final goal:
# Efficiently find hyperparameters that maximize validation accuracy.
#
#
# -----------------------------------------------------------------------------
# 🔥 One-line memory
# -----------------------------------------------------------------------------
#
# Random Search → blind guessing
#
# Optuna →
# learn graph + use uncertainty + move toward peak
#
# =============================================================================

In [ ]:
# TPE (Tree-structured Parzen Estimator) models two distributions:
# one for good hyperparameter values and one for bad hyperparameter values.
# It chooses new hyperparameters that are likely to belong to the good region
# and unlikely to belong to the bad region.

In [ ]:
# "Tree-structured" in TPE means the hyperparameter space can have
# conditional branches (e.g., if optimizer=Adam tune beta1/beta2,
# if optimizer=SGD tune momentum). This creates a tree-like dependency
# structure, and TPE searches efficiently through those branches.

In [ ]:
# Objective function defines one trial:
# choose hyperparameters → build model → train/evaluate → return metric
# Optuna repeatedly calls this function and tries to optimize returned value.

In [ ]:
# Study stores all trial history and results.
# Sampler (TPE) analyzes that history and decides the next hyperparameter values.
# trial.suggest_int()/suggest_float() fetch those chosen values for the current trial.
# objective() trains/evaluates model, and study stores the new result.

In [ ]:
'''
def objective(trial):

    # 1) suggest hyperparameters
    hp1 = trial.suggest_...
    hp2 = trial.suggest_...

    # 2) create model
    model = ...

    # 3) evaluate
    score = ...

    # 4) return metric
    return score

study = optuna.create_study(direction="maximize")
study.optimize(objective,n_trials=50)
'''

In [15]:
class mynn(nn.Module):

  def __init__(self,input_dim,output_dim,num_hidden_layers,neurons_per_layer,dropout_rate):

    super().__init__()

    layers = []

    for i in range(num_hidden_layers):

      layers.append(nn.Linear(input_dim,neurons_per_layer))
      layers.append(nn.BatchNorm1d(neurons_per_layer))
      layers.append(nn.ReLU())
      layers.append(nn.Dropout(dropout_rate))
      input_dim = neurons_per_layer


    layers.append(nn.Linear(neurons_per_layer,output_dim))

    self.model = nn.Sequential(*layers)

  def forward(self,features):

    out=self.model(features)

    return out

In [16]:
# *layers unpacks the list into separate arguments
# nn.Sequential(*layers) means:
# nn.Sequential(layer1, layer2, layer3, ...)

In [17]:
def objective(trail):

  #hyperparameter search space
                                        #name of the layer,starting_num,final_num
  num_hidden_layers = trail.suggest_int("num_hidden_layers",1,5)
  neurons_per_layer = trail.suggest_int("neurons_per_layers",8,128,step=8)
  epochs = trail.suggest_int('epochs',10,50,step=10)
  learning_rate = trail.suggest_float('learning rate',1e-5,1e-1,log=True)
  dropout_rate = trail.suggest_float('dropout_rate',0.1,0.5,step=0.1)
  batch_size = trail.suggest_categorical('batch_size',[16,32,64,128])
  optimizer_name = trail.suggest_categorical('optimizer',['Adam','SGD','RMSprop'])
  weight_decay = trail.suggest_float('weight_decay',1e-5,1e-3,log=True)


  train_loader = DataLoader(train_dataset , batch_size = batch_size,shuffle = True,pin_memory = True)
  test_loader = DataLoader(test_dataset,batch_size = batch_size,shuffle=False,pin_memory=True)

  #model initstialisation
  input_dim=784
  output_dim=10

  model=mynn(input_dim,output_dim,num_hidden_layers,neurons_per_layer,dropout_rate)
  model.to(device)


  #optimizer selection
  criterion = nn.CrossEntropyLoss()
  if optimizer_name == 'Adam':
    optimizer=optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
  elif optimizer_name == 'SGD':
    optimizer=optim.SGD(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
  else:
    optimizer=optim.RMSprop(model.parameters(), lr=learning_rate, weight_decay=weight_decay)



  #training loop

  for i in range(epochs):

    for batch_features,batch_labels in train_loader:

      batch_features,batch_labels = batch_features.to(device),batch_labels.to(device)

      #forward pass
      output=model(batch_features)

      #loss
      loss = criterion(output,batch_labels)

      #back pass
      optimizer.zero_grad()
      loss.backward()

      #update gradeints
      optimizer.step()

# evaluation
  model.eval()
  # evaluation on test data
  total = 0
  correct = 0

  with torch.no_grad():

    for batch_features, batch_labels in test_loader:

      # move data to gpu
      batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

      outputs = model(batch_features)

      predicted = torch.argmax(outputs,dim=1)

      total = total + batch_labels.shape[0]

      correct = correct + (predicted == batch_labels).sum().item()

    accuracy = correct/total

  return accuracy



In [18]:
import optuna

study = optuna.create_study(direction='maximize')#definig the study and sampler here

[I 2026-04-29 04:15:49,548] A new study created in memory with name: no-name-1f5eb2bf-0b3a-4bc0-9871-12ea7944e485


In [19]:
study.optimize(objective, n_trials=10)

[I 2026-04-29 04:16:59,030] Trial 0 finished with value: 0.88375 and parameters: {'num_hidden_layers': 5, 'neurons_per_layers': 96, 'epochs': 20, 'learning rate': 0.0006702787642771744, 'dropout_rate': 0.30000000000000004, 'batch_size': 64, 'optimizer': 'Adam', 'weight_decay': 1.7727994964004905e-05}. Best is trial 0 with value: 0.88375.
[I 2026-04-29 04:17:42,694] Trial 1 finished with value: 0.6893333333333334 and parameters: {'num_hidden_layers': 1, 'neurons_per_layers': 120, 'epochs': 50, 'learning rate': 1.9471091905462767e-05, 'dropout_rate': 0.5, 'batch_size': 128, 'optimizer': 'SGD', 'weight_decay': 0.000727426822697493}. Best is trial 0 with value: 0.88375.
[I 2026-04-29 04:18:30,680] Trial 2 finished with value: 0.8794166666666666 and parameters: {'num_hidden_layers': 1, 'neurons_per_layers': 120, 'epochs': 30, 'learning rate': 0.00014021001635523782, 'dropout_rate': 0.5, 'batch_size': 64, 'optimizer': 'RMSprop', 'weight_decay': 0.0005880462312684853}. Best is trial 0 with va

In [20]:
print("Best trial:")
print(f"  Value: {study.best_trial.value}")
print("  Params: ")
for key, value in study.best_trial.params.items():
    print(f"    {key}: {value}")

Best trial:
  Value: 0.8880833333333333
  Params: 
    num_hidden_layers: 3
    neurons_per_layers: 128
    epochs: 50
    learning rate: 0.06520178254660235
    dropout_rate: 0.1
    batch_size: 64
    optimizer: SGD
    weight_decay: 0.00019770256147699254
